Mount Google Drive and initialize project

In [1]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Project configuration
PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

# Set working directory
os.chdir(PROJECT_DIR)

print("Project mounted successfully!")
print("Working directory:", os.getcwd())
print("Python version:", sys.version)

Mounted at /content/drive
Project mounted successfully!
Working directory: /content/drive/MyDrive/DumLLM
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


#GitHub Auto Sync

In [ ]:
import os
import sys
import subprocess
import base64
from getpass import getpass
from google.colab import userdata

# ==========================================
# DumLLM - GitHub Auto Sync
# ==========================================

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
REPO_URL = "https://github.com/ShubhamKumarSahu-svg/DumLLM.git"
BRANCH = "main"

os.chdir(PROJECT_DIR)

def run(command, check=True):
    return subprocess.run(
        command,
        check=check,
        text=True,
        capture_output=True
    )

# 1. Verify Git repository
if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    raise RuntimeError("Git repository not found!")

# 2. Configure remote
remotes = run(["git", "remote"]).stdout.split()

if "origin" not in remotes:
    run(["git", "remote", "add", "origin", REPO_URL])
else:
    run(["git", "remote", "set-url", "origin", REPO_URL])

# 3. Protect unnecessary files
gitignore = os.path.join(PROJECT_DIR, ".gitignore")

ignore_rules = [
    ".venv/",
    "venv/",
    "__pycache__/",
    "*.py[cod]",
    ".env",
    ".env.*",
    "!.env.example",
    ".ipynb_checkpoints/",
]

existing = ""
if os.path.exists(gitignore):
    with open(gitignore, "r") as f:
        existing = f.read()

with open(gitignore, "a") as f:
    for rule in ignore_rules:
        if rule not in existing.splitlines():
            f.write(f"\n{rule}")

# Remove virtual environment from Git tracking if previously added
tracked = run(
    ["git", "ls-files", "--", ".venv", "venv"],
).stdout.strip()

if tracked:
    run(["git", "rm", "-r", "--cached", "--", ".venv", "venv"])

# 4. Get commit message
commit_message = input("Enter commit message: ").strip()

if not commit_message:
    raise ValueError("Commit message cannot be empty!")

# 5. Stage all changes
run(["git", "add", "-A"])

# 6. Check for changes
status = run(["git", "diff", "--cached", "--quiet"], check=False)

if status.returncode == 0:
    print("No changes to commit. Everything is up to date!")
else:
    # 7. Commit
    run(["git", "commit", "-m", commit_message])
    print("Commit created successfully!")

    # 8. Get token from Colab Secrets
    token = userdata.get("GITHUB_TOKEN")

    if not token:
        raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets!")

    auth = base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()

    # 9. Push securely without saving credentials
    result = subprocess.run(
        [
            "git",
            "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
            "push", "-u", "origin", f"HEAD:{BRANCH}"
        ],
        cwd=PROJECT_DIR,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError("GitHub push failed. Check authentication and branch.")

    print("\nSuccessfully pushed to GitHub!")
    print(f"Repository: {REPO_URL}")
    print(f"Branch: {BRANCH}")
    print(f"Commit: {commit_message}")

print("\nDumLLM GitHub sync complete!")